# 🔀 Notebook 3: Motion Control — Position vs. Velocity Modes

This notebook builds intuition for the two control modes exposed by `/wmx/axis/set_mode`, by executing one motion command in each mode and comparing `actual_pos` before and after — so the difference shows up in real numbers, not just in theory.

### 🎯 Objective & Scope
* **Focus**: Runs Method A (a single move in Position mode) and Method B (a single jog in Velocity mode), reading the axis state around each so the two behaviors can be compared directly.
* **Prerequisite**: Notebook 1 (WMX engine + EtherCAT communication) and Notebook 2 (axis power-on, homing, Position mode) must be completed and still running.

### 🔄 Comparison Sequence
* **Method A: Position Mode Move**: Command an absolute target position and watch the axis travel there and stop. Reuses the Position mode already set in Notebook 2's Step 8.
* **Step 9: Set Control Mode (Velocity)**: Re-confirm a clean state (clear alarm → servo on → homing) and switch the axes into Velocity mode ahead of the jog test.
* **Step 10: Method B: Velocity Mode Move**: Command a target speed for a fixed duration and watch continuous rotation with no destination.
* **Step 11: Set Control Mode (Position)**: Return the axes to Position mode so later notebooks (e.g. Notebook 4's manipulator trajectory control) start from a known state.

#### 🔌 Python ROS 2 Client Connection

As in Notebook 1, this creates a new `WmxClient node` (`wmx_client_03`). See the markdown above the corresponding cell in Notebook 1 for details on the cell structure. Since this notebook needs to poll `actual_pos` repeatedly, the state subscription and position-reading logic has been split out into the `PositionReader` helper in `wmx_motion_utils.py`, following the same pattern as `WmxClient` in `wmx_utils.py`.

In [2]:
import time
import rclpy
from wmx_r2_message.srv import SetAxis
from wmx_r2_message.msg import AxisPose, AxisVelocity
from wmx_utils import WmxClient
from wmx_motion_utils import PositionReader

if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_client_03')
print(f"✅ WMX Client Ready for Notebook 03. Target Axes: {wmx.axis_list}")

pos_reader = PositionReader(wmx)

✅ WMX Client Ready for Notebook 03. Target Axes: [0, 1]


## Method A: Position Mode Move

Position mode commands are **absolute**: `target` is a coordinate to go to, not a distance to travel.

That coordinate system's zero point is defined by homing (Notebook 2, Step 7) — whatever position the axis physically happens to be in when `/wmx/axis/homing` runs becomes `0`. From there, `actual_pos` is the total rotation (in radians, given this project's gear ratio setup) measured from that zero point. There's no wraparound — since soft limits are disabled in this config, the number just keeps growing (or shrinking) the more you rotate the same direction.

So sending `target = 10.0` means "go to the point ~1.6 turns away from wherever homing was done," regardless of where the axis currently is.

> ℹ️ The axes are already in Position mode from Notebook 2's Step 8, so this section moves straight to the command — the clear-alarm → servo-on → homing → set-mode sequence isn't repeated here.

In [6]:
axes = wmx.axis_list
zeros = [0] * len(axes)

# 1. Clear alarm
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
print(f"1) Clear Alarm: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 2. Servo On
req = SetAxis.Request()
req.index = axes
req.data = [0] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"2) Servo On: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 3. Homing
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/homing', req)
print(f"3) Homing: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 4. Set Position mode (0 = Position)
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"4) Set Position Mode: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

print("\n🚀 Ready. Run the next cell to move once in Position mode.")

1) Clear Alarm: ✅ Cleared alarm axis 0; Cleared alarm axis 1; 
2) Servo On: ✅ Set axis 0 off; Set axis 1 off; 
3) Homing: ❌ Failed to home axis 0. Error=1567 (WMX3Api Error: The servos of one or more commanded axes are off.); Failed to home axis 1. Error=1567 (WMX3Api Error: The servos of one or more commanded axes are off.); 
4) Set Velocity Mode: ✅ Set axis 0 in Position mode; Set axis 1 in Position mode; 

🚀 Ready. Run the next cell to move once in Velocity mode (Method B).


In [4]:
TARGET_POSITION = 10.0    # target position (radians)
MOVE_VELOCITY   = 5.0     # move speed
MOVE_ACC        = 10.0    # acceleration
MOVE_DEC        = 10.0    # deceleration
WAIT_SEC        = 3.0     # seconds to wait for the move to finish

pos_pub = wmx.create_publisher(AxisPose, '/wmx/axis/position', 1)

before = pos_reader.get_current_position()
print(f"Position before move: {before}")

msg = AxisPose()
msg.index = wmx.axis_list
msg.target = [TARGET_POSITION] * len(wmx.axis_list)
msg.velocity = [MOVE_VELOCITY] * len(wmx.axis_list)
msg.acc = [MOVE_ACC] * len(wmx.axis_list)
msg.dec = [MOVE_DEC] * len(wmx.axis_list)

print(f"🎯 [Method A] Moving to target position {TARGET_POSITION} ...")
pos_pub.publish(msg)
time.sleep(WAIT_SEC)

after = pos_reader.get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"Position after move: {after}")
print(f"Distance traveled (rad): {[f'{t:.3f}' for t in traveled]}")
print(f"Distance traveled (turns): {[f'{t:.3f}' for t in turns]}")

Position before move: [-1007.1732325553894, 507.1367405653]
🎯 [Method A] Moving to target position 10.0 ...
Position after move: [-994.0434973239899, 494.0077518224716]
Distance traveled (rad): ['13.130', '-13.129']
Distance traveled (turns): ['2.090', '-2.090']


## Method B — Velocity Mode Move

Velocity mode has no target coordinate at all — you command a **speed**, and the axis keeps spinning at that speed until you explicitly command `0`. There's no "arrival," so it's inherently a **relative** motion: how far the axis travels depends on how long you let it run (roughly `velocity × time`, plus the accel/decel ramps).

Unlike Method A, there's no destination to compute ahead of time — you only find out how far it went by comparing `actual_pos` before and after.

In [ ]:
axes = wmx.axis_list
zeros = [0] * len(axes)

# 1. Clear alarm
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
print(f"1) Clear Alarm: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 2. Servo On
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"2) Servo On: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 3. Homing
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/homing', req)
print(f"3) Homing: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 4. Set Velocity mode (1 = Velocity)
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"4) Set Velocity Mode: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

print("\n🚀 Ready. Run the next cell to move once in Velocity mode.")

In [ ]:
RUN_VELOCITY = 3.0    # target speed
RUN_ACC      = 10.0   # acceleration
RUN_DEC      = 10.0   # deceleration
RUN_SEC      = 3.0    # how long to run at this speed (seconds)

vel_pub = wmx.create_publisher(AxisVelocity, '/wmx/axis/velocity', 1)

before = pos_reader.get_current_position()
print(f"Position before move: {before}")

msg = AxisVelocity()
msg.index = wmx.axis_list
msg.velocity = [RUN_VELOCITY] * len(wmx.axis_list)
msg.acc = [RUN_ACC] * len(wmx.axis_list)
msg.dec = [RUN_DEC] * len(wmx.axis_list)

print(f"🚀 [Method B] Running at velocity {RUN_VELOCITY} for {RUN_SEC}s ...")
vel_pub.publish(msg)
time.sleep(RUN_SEC)

# Stop (velocity = 0)
msg.velocity = [0.0] * len(wmx.axis_list)
vel_pub.publish(msg)
print("🛑 Stopped.")

after = pos_reader.get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"Position after move: {after}")
print(f"Distance traveled (rad): {[f'{t:.3f}' for t in traveled]}")
print(f"Distance traveled (turns): {[f'{t:.3f}' for t in turns]}")
print(f"For reference, velocity x time = {RUN_VELOCITY * RUN_SEC:.2f} rad (theoretical, excludes accel/decel ramps)")

## (Optional) Motor Servo Off

Disabling the servo (`set_on` with `data = 0`) cuts power to the motor and releases the holding torque, so the axis becomes free to move by hand. Run this only if you're completely done testing for the session — if you plan to continue straight on to Notebook 4, leave the servo on so you don't have to redo Notebook 2's homing (Step 7) before your next position-based test.

In [5]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"Servo Off: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

'\nRUN_SHUTDOWN = False\n\nif RUN_SHUTDOWN:\n    req = SetAxis.Request()\n    req.index = wmx.axis_list\n    req.data = [0] * len(wmx.axis_list)\n    res = wmx.call(SetAxis, \'/wmx/axis/set_on\', req)\n    print(f"Servo Off: {\'✅ \' + res.message if res.success else \'❌ \' + res.message}")\n'